# 实验 10 — dlt schema contract + 列类型 hints

**目标**：把 dlt 的 schema 自动演化从默认的 “evolve everything” 变成可控的契约（contract），并显式约定列类型。

**为什么生产上要管这事**：
- 上游 API 加了新字段，自动建列也许不是你想要的（脏数据、PII 泄漏）
- 列类型推断在小样本里可能蒙错（int → bigint 但首批都是小数）
- 下游 dbt 模型依赖列类型，schema drift 会引发 cast 失败

## schema_contract 的三种模式

| 模式 | 含义 |
|---|---|
| `evolve` | 默认。新列/新表/新类型都允许 |
| `freeze` | 任何新增/变更都报错，强制人审 |
| `discard_row` / `discard_value` | 静默丢弃违规行/列 |

可在三个粒度上设：`tables`（整张表）、`columns`（新增列）、`data_type`（已有列类型变化）。

In [ ]:
# 1) 显式列类型 hint：把 rate 固定为 decimal，而不是 dlt 推断的 double
import dlt, requests, datetime
from typing import Iterator

@dlt.resource(
    write_disposition='merge', primary_key=['date','currency'], name='daily_rates',
    columns={
        'date': {'data_type': 'date', 'nullable': False},
        'currency': {'data_type': 'text', 'nullable': False},
        'rate': {'data_type': 'decimal', 'precision': 18, 'scale': 6, 'nullable': False},
    }
)
def daily_rates() -> Iterator[dict]:
    url = 'https://api.frankfurter.app/2024-12-01..2024-12-10?to=USD,GBP'
    data = requests.get(url, timeout=30).json()
    for d, rates in data['rates'].items():
        for cur, rate in rates.items():
            yield {'date': d, 'currency': cur, 'rate': rate}

pipe = dlt.pipeline(
    pipeline_name='ecb_contract_demo',
    destination=dlt.destinations.duckdb('../data/sandbox.duckdb'),
    dataset_name='raw_contract',
)
print(pipe.run(daily_rates()))

In [ ]:
# 验证 rate 是 decimal(18,6) 而不是 double
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
con.execute('describe raw_contract.daily_rates').df()

In [ ]:
# 2) schema_contract='freeze': 加一个意外字段应该报错
@dlt.resource(
    write_disposition='merge', primary_key=['date','currency'], name='daily_rates',
    schema_contract={'columns': 'freeze'},
)
def daily_rates_with_extra() -> Iterator[dict]:
    yield {'date':'2024-12-11','currency':'USD','rate':1.05,'unexpected_field':'oops'}

try:
    pipe.run(daily_rates_with_extra())
    print('（没报错？检查 dlt 版本）')
except Exception as e:
    print('CAUGHT:', type(e).__name__)
    print(str(e)[:600])

In [ ]:
# 3) schema_contract='discard_value': 多余字段被丢弃，行还是写进去
@dlt.resource(
    write_disposition='merge', primary_key=['date','currency'], name='daily_rates',
    schema_contract={'columns': 'discard_value'},
)
def daily_rates_discard() -> Iterator[dict]:
    yield {'date':'2024-12-11','currency':'USD','rate':1.05,'unexpected_field':'silently_dropped'}

print(pipe.run(daily_rates_discard()))
print()
print(con.execute('describe raw_contract.daily_rates').df())
print()
print("如果 unexpected_field 没有出现在列里，就说明 dlt 默默丢弃了它")

## 生产环境踩坑提示

- **`evolve` 是开发友好但治理灾难**：上游加 PII 字段，你的 raw 表立刻就有了。生产建议默认 `freeze`，要新增字段走代码审。
- **`columns={}` 的 hint 在首次写入时生效**。列已存在再改类型不一定立刻生效，要 drop 表或显式 `apply_hints()`。
- **schema 持久化在 `_dlt_version` 表**，PR 里可以 diff 出来。把 `pipeline.schema.to_pretty_yaml()` 写到一个 git-tracked 文件，做 schema-as-code review。
- **`data_type` 变更模式**：上游突然把 int 改成 string，`freeze` 会报错，`discard_row` 会丢这一行。哪种更安全取决于业务。

## 思考题

- 把 `schema_contract={'tables': 'freeze'}` 加到 source 上、然后 yield 一个新表名，看 dlt 怎么处理。
- `columns: 'freeze'` 和 `data_type: 'freeze'` 有什么区别？哪个对应“不允许新增列”、哪个对应“已有列不许变类型”？
- 怎么把当前 schema 序列化为 YAML 放进代码库做 review？（`pipe.default_schema.to_pretty_yaml()`）